# 第1章：GPT 架构拆解

## 本章目标
- 理解 GPT (Generative Pre-trained Transformer) 的完整架构
- 从零手写每个组件：CausalSelfAttention、MLP、Block、GPT
- 理解 GPT-2 的参数配置和计算量分析

## 前置知识
- Transformer 的 Self-Attention 机制（你知道 Q/K/V 是什么）
- PyTorch nn.Module 的基本用法
- 矩阵乘法和 softmax 运算

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch tiktoken matplotlib
    !wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## GPT 架构速览

GPT = Token Embedding + Positional Embedding + N × Transformer Block + LayerNorm + Linear Head

与 BERT 的区别：GPT 是 **decoder-only**，使用 causal mask（下三角矩阵）确保只能看到当前和之前的 token。

参考论文：[Attention Is All You Need](https://arxiv.org/abs/1706.03762), [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention.

    与标准 Self-Attention 的区别：
    1. 多头并行计算 (n_head 个独立的 attention head)
    2. Causal mask：只关注当前位置及之前的 token
    """
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)  # Q, K, V 合并计算
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        # causal mask: 下三角矩阵
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

分析 Attention 的计算过程：

- **Q (Query)**: "我在找什么？" — shape: `(B, n_head, T, head_dim)`
- **K (Key)**: "我有什么？" — shape: `(B, n_head, T, head_dim)`
- **V (Value)**: "我的内容是什么？" — shape: `(B, n_head, T, head_dim)`
- **Attention weights**: `softmax(Q @ K^T / sqrt(d))` — shape: `(B, n_head, T, T)`
- **Output**: `weights @ V` — shape: `(B, n_head, T, head_dim)` → reshape → `(B, T, n_embd)`

Causal mask 确保位置 i 只能看到位置 0..i 的信息。

In [ ]:
class MLP(nn.Module):
    """Feed-forward network with GELU activation.

    标准做法：d_model → 4 × d_model → d_model
    GPT-2 使用 GELU 而不是 ReLU。
    """
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

In [ ]:
class Block(nn.Module):
    """Pre-norm Transformer Block.

    GPT 使用 Pre-LN（LayerNorm 在 attention/MLP 之前），
    而原始 Transformer 使用 Post-LN。
    """
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # residual connection
        x = x + self.mlp(self.ln_2(x))     # residual connection
        return x

## Pre-LN vs Post-LN

**Post-LN**（原始 Transformer）：`x = LayerNorm(x + Sublayer(x))`
**Pre-LN**（GPT-2 及之后的主流做法）：`x = x + Sublayer(LayerNorm(x))`

Pre-LN 的优势：
1. 训练更稳定（梯度流更平滑）
2. 不需要 learning rate warmup（虽然 GPT-2 还是用了 warmup）
3. 大多数现代 LLM 都采用 Pre-LN

In [ ]:
class GPTConfig:
    """GPT-2 的配置参数"""
    def __init__(self, vocab_size=50304, block_size=1024,
                 n_layer=12, n_head=12, n_embd=768, dropout=0.1):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.block_size = config.block_size
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)    # token embedding
        self.wpe = nn.Embedding(config.block_size, config.n_embd)    # position embedding
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config.n_embd, config.n_head,
                                       config.block_size, config.dropout)
                                 for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying: embedding 和 output head 共享权重
        self.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.wte(idx)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
# 用 baby-GPT 配置测试（小到可以 CPU 跑）
config = GPTConfig(vocab_size=65, block_size=128, n_layer=4, n_head=4, n_embd=128)
model = GPT(config)
print(f"参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# forward pass 测试
x = torch.randint(0, config.vocab_size, (2, 64))  # batch=2, seq_len=64
logits, loss = model(x, targets=x)
print(f"logits shape: {logits.shape}")
print(f"loss: {loss.item():.4f}")

## GPT-2 参数量分析

| 配置 | n_layer | n_head | n_embd | 参数量 |
|------|---------|--------|--------|--------|
| GPT-2 Small | 12 | 12 | 768 | ~124M |
| GPT-2 Medium | 24 | 16 | 1024 | ~350M |
| GPT-2 Large | 36 | 20 | 1280 | ~774M |
| GPT-2 XL | 48 | 25 | 1600 | ~1.6B |

参数主要花在哪里？ embedding + N × (attention + MLP) + output head

In [ ]:
configs = {
    "GPT-2 Small":  GPTConfig(n_layer=12, n_head=12, n_embd=768),
    "GPT-2 Medium": GPTConfig(n_layer=24, n_head=16, n_embd=1024),
    "GPT-2 Large":  GPTConfig(n_layer=36, n_head=20, n_embd=1280),
    "GPT-2 XL":     GPTConfig(n_layer=48, n_head=25, n_embd=1600),
}
for name, cfg in configs.items():
    model = GPT(cfg)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"{name:15s}: {params:.0f}M params")

## 练习

1. 修改 `GPTConfig` 的 `n_layer` 和 `n_embd`，观察参数量如何变化
2. 打印 `model.h[0].attn.bias` 的 shape，理解 causal mask 是如何存储的
3. 尝试用 `torch.nn.functional.scaled_dot_product_attention` 替换手写的 attention（PyTorch 2.0+ 的 Flash Attention）

## 延伸阅读

- [nanoGPT model.py](https://github.com/karpathy/nanoGPT/blob/master/model.py) — 本章节的参考实现
- [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/) — 可视化 GPT-2 架构
- [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)